# Fusión Sensorial + Índice de Valor (PySpark) — Colab

Ejecuta las celdas en orden. Resumen técnico completo en `docs/RESUMEN_PROYECTO.txt`.

## 1. Preparar entorno y código

In [ ]:
!java -version
!pip -q install pyspark Pillow
!git clone -q https://github.com/JoseMiguelFonsecaCordoba/fusion_sensorial_bigdata.git 2>/dev/null || git -C fusion_sensorial_bigdata pull -q
%cd /content/fusion_sensorial_bigdata
!ls

## 2. Subir el dataset
Selecciona `Rest-Mex_2025_train_clean.csv` (≈61 MB). *Opcional:* el repo ya trae `data/processed/profiles.json`, así que puedes saltarte los pasos 2 y 3 si no vas a recalcular.

In [ ]:
from google.colab import files
import shutil
files.upload()
shutil.move("Rest-Mex_2025_train_clean.csv", "data/raw/")
!ls -l data/raw

## 3. Big Data: reseñas → perfiles (PySpark)

In [ ]:
!python -m src.build_profiles

## 4. Evaluación
Coherencia, presupuesto, cobertura, tabla por preset y fiabilidad por mitades (esta última usa Spark y el CSV).

In [ ]:
!python -m src.evaluate --presets --split-half

## 5. Demos para la presentación
También: `python -m src.demo` corre las tres con narrativa.

In [ ]:
# Fiesta: reggaeton + rojo/naranja
!python -m src.main --preset reggaeton --color "#FF5733" --top 5

In [ ]:
# Calma: música lenta y melancólica + azul oscuro
!python -m src.main --bpm 70 --energy 0.2 --danceability 0.1 --valence 0.25 --color "#1A365D" --top 3

In [ ]:
# Joyas ocultas con presupuesto bajo
!python -m src.main --color "#2ECC71" --presupuesto 0.9 --solo-joyas --top 3

## 6. Consola interactiva con narrativa
Usa directamente `src.sensory` y `src.recommender` (misma matemática que la CLI y el front web).

In [ ]:
# @title Consola sensorial
Preset = "ninguno" # @param ["ninguno", "reggaeton", "pop", "rock", "electronica", "salsa", "cumbia", "regional", "metal", "jazz", "bolero", "lofi", "clasica", "ambient"]
BPM = 140 # @param {type:"slider", min:60, max:180, step:5}
Energia = 0.85 # @param {type:"slider", min:0.0, max:1.0, step:0.05}
Bailabilidad = 0.8 # @param {type:"slider", min:0.0, max:1.0, step:0.05}
Valencia = 0.75 # @param {type:"slider", min:0.0, max:1.0, step:0.05}
Color_Hex = "#1F6966" # @param {type:"string"}
Presupuesto = 0.5 # @param {type:"slider", min:0.0, max:1.0, step:0.05}
Solo_Joyas = False # @param {type:"boolean"}

from src import demo

demo.run(demo.recommender.load_profiles(), "Consola sensorial",
         preset=None if Preset == "ninguno" else Preset,
         music=(BPM, Energia, Bailabilidad, Valencia), color=Color_Hex,
         presupuesto=Presupuesto, solo_joyas=Solo_Joyas)

## 7. Front web
En tu computadora: `python -m src.web` y abre http://localhost:8000. (En Colab no hay navegador local; usa la consola de arriba.)

## 8. Descargar los perfiles recalculados

In [ ]:
from google.colab import files
files.download("data/processed/profiles.json")